In [37]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
!pip install evidently

In [39]:
import numpy as np
import math
import pandas as pd
import glob
import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from dateutil.relativedelta import relativedelta
import xgboost as xgb
from scipy.stats import ks_2samp
from statsmodels.stats.multitest import multipletests

from evidently import Report
from evidently.presets import DataDriftPreset


# Statistical Drift

Statistical drift is when the underlying properties of dataset change and become statistically different then what was seen previously. In practice, a change in statistical properties of a dataset mean that prior assumption and training of a model might no longer hold up when looking at this new dataset, thus retraining will need to happen.

There is 3 main types of drift: Data Drift (Covariate Shift), Concept Drift and Prior Probability Drift(Target Drift)

Data Drift occurs when distribution of the input data changes over time but the underlying relationship between input(X) and output(Y) stays the same

Concept Drift occurs when the task that the model is given itself changes, meaning that the relationship between X and Y changes. Example if model was told to detect spam, but the type of spam changed significantly

Target Shift occurs when distribution of Target changes, but in reality we dont test this in production because we dont know the Target values of an upcoming month, this is what the model is predicting for.


In [40]:
start_train_date = "2025-01-01T00:00:00.000000"


datasetPath = datasetPath = "/content/drive/MyDrive/Colab Notebooks/nyc-taxi-drift-forecasting/data/aggregatedData.csv"
Catcolumns = ['day_factor', 'PULocationID']
df = pl.read_csv(datasetPath)
df = df.with_columns(
    pl.col("tpep_pickup_datetime").str.to_datetime("%Y-%m-%dT%H:%M:%S%.f")
)
df = df.sort("tpep_pickup_datetime")
df.head()



tpep_pickup_datetime,PULocationID,trip_count,hour,day_of_month,year,month,is_weekend,day_factor
datetime[μs],i64,i64,i64,i64,i64,i64,bool,str
2025-01-01 00:00:00,22,3,0,1,2025,1,false,"""Wednesday"""
2025-01-01 00:00:00,40,4,0,1,2025,1,false,"""Wednesday"""
2025-01-01 00:00:00,14,4,0,1,2025,1,false,"""Wednesday"""
2025-01-01 00:00:00,218,3,0,1,2025,1,false,"""Wednesday"""
2025-01-01 00:00:00,129,4,0,1,2025,1,false,"""Wednesday"""


In [41]:
#Hour x Location Interaction Feature

df = df.with_columns(
    (pl.col('PULocationID').cast(str) + '_' + pl.col('hour').cast(str)).alias('loc_hour_key')
)

df = df.sort(['PULocationID', 'tpep_pickup_datetime'])

# 3. Create the 1-hour and 2-hour lag features using window functions globally
df = df.with_columns([
    pl.col('trip_count').shift(1).over('PULocationID').alias('trip_count_lag_1h'),
    pl.col('trip_count').shift(2).over('PULocationID').alias('trip_count_lag_2h')
])


# Drift Analysis

In [42]:
def covariate_shift_detect(df_base, df_test):
  #significance level
  alpha = 0.05

  df_base = df_base.to_pandas()
  df_test = df_test.to_pandas()
  result = []
  com_numeric_col = (df_base.select_dtypes(include = [np.number])
  .columns.intersection(df_test.select_dtypes(include = [np.number]).columns)
  )
  # Explicitly drop target, dates, and ID features
  colIgnore = ['trip_count', 'month', 'year', 'day_of_month', 'PULocationID']

  features = []
  ks_distances = []
  p_values = []
  feature_drifted = []

  # Set a practical distance threshold instead of relying on p-value
  # 0.05 to 0.10 is standard for large-sample drift detection
  DISTANCE_THRESHOLD = 0.05

  for col in com_numeric_col:
      if col not in colIgnore:
          sample1 = df_base[col].dropna()
          sample2 = df_test[col].dropna()

          if len(sample1) == 0 or len(sample2) == 0:
              continue

          # Perform K-S Test
          stat, p = ks_2samp(sample1, sample2)

          features.append(col)
          ks_distances.append(stat)
          p_values.append(p)

          # Flag drift based on distance, NOT p-value
          feature_drifted.append(stat > DISTANCE_THRESHOLD)

  report = pd.DataFrame({
      "Feature": features,
      "K-S Distance (D)": ks_distances,
      "Raw p-value": p_values,
      "Feature Drifted": feature_drifted
  })

  drifted_features_count = report['Feature Drifted'].sum()

  # Guard against division by zero if no valid features remain
  pct_drifted = drifted_features_count / len(features) if features else 0.0

  # Dataset is considered drifted if more than 20% of your operational features genuinely drift
  dataset_drifted = pct_drifted > 0.20

  return {
      "dataset_drifted": dataset_drifted,
      "percentage_features_drifted": round(pct_drifted * 100, 2),
      "feature_report": report
  }



In [43]:
#Concept Drift

def concept_drift_with_warmup(train_res, test_res, delta=0.005, lambda_thresh=50.0):
    # Establish the initial baseline using historical training errors
    abs_train_errors = np.abs(train_res)
    running_mean = np.mean(abs_train_errors)

    # Initialize the sum tracker
    sum_m = 0.0
    min_sum_m = float('inf')

    drift_detected = False
    drift_index = -1

    abs_test_errors = np.abs(test_res)

    # Start checking the test errors with historical context intact
    for t, error in enumerate(abs_test_errors):
        # Update the mean using a continuous sample count
        # (len(train_res) prevents a few early test points from skewing the mean)
        total_samples = len(abs_train_errors) + t + 1
        running_mean = running_mean + (error - running_mean) / total_samples

        # Accumulate the deviations
        sum_m += (error - running_mean - delta)

        if sum_m < min_sum_m:
            min_sum_m = sum_m

        if (sum_m - min_sum_m) > lambda_thresh:
            drift_detected = True
            drift_index = t
            break

    return drift_detected, drift_index

In [44]:


start_train_date = datetime(2025, 1, 1)
end_train_date = datetime(2025, 12, 31)

window_size = 4
forecast_size = 1

results = []
metrics_summary = []
current_train_start = start_train_date



reports = pd.DataFrame(columns = ["BaseMonths","TestMonths","Report" ])

while True:
  current_train_end = current_train_start + relativedelta(months=window_size)
  current_test_end = current_train_end + relativedelta(months=forecast_size)

  if current_test_end > end_train_date:
    break



  train_df = df.filter(
      (pl.col("tpep_pickup_datetime") >= current_train_start) &
      (pl.col("tpep_pickup_datetime") < current_train_end)
  )

  test_df = df.filter(
      (pl.col("tpep_pickup_datetime") >= current_train_end) &
      (pl.col("tpep_pickup_datetime") < current_test_end)
  )



  loc_hour_avg = (
    train_df.group_by('loc_hour_key')
    .agg(pl.col('trip_count').mean().alias('loc_hour_avg'))
  )
  global_avg = train_df['trip_count'].mean()


  train_df = train_df.join(loc_hour_avg, on='loc_hour_key', how='left')
  test_df = test_df.join(loc_hour_avg, on='loc_hour_key', how='left')
  test_df = test_df.with_columns(pl.col('loc_hour_avg').fill_null(global_avg))

  train_df = train_df.with_columns([
    pl.col('trip_count_lag_1h').fill_null(pl.col('loc_hour_avg')),
    pl.col('trip_count_lag_2h').fill_null(pl.col('loc_hour_avg'))
  ])

  test_df = test_df.with_columns([
    pl.col('trip_count_lag_1h').fill_null(pl.col('loc_hour_avg')),
    pl.col('trip_count_lag_2h').fill_null(pl.col('loc_hour_avg'))
  ])


  train_df = train_df.drop(['loc_hour_key'])
  test_df = test_df.drop(['loc_hour_key'])


  #Data Drift

  #Kolmogorov-Smirnov (K-S) Test
  #Null Hypothesis being that the 2 datasets come from the same distribution
  KSTest = covariate_shift_detect(train_df, test_df)

  if KSTest["dataset_drifted"]:
    print(f"There has been a data drift with {KSTest["percentage_features_drifted"]}% of the features thus a covariate shift")
    print("Occured when comparing base months of " + current_train_start.strftime("%B") + "-" + (current_train_end - relativedelta(months=1)).strftime("%B") + " to test month of " + current_train_end.strftime("%B"))
    print(KSTest["feature_report"])

  #Page Hinkley Method
  pHTest = Report([DataDriftPreset()], include_tests= True)
  eval = pHTest.run(current_data = test_df.to_pandas().drop(columns = ['trip_count', 'month', 'year', 'day_of_month','PULocationID']), reference_data = train_df.to_pandas().drop(columns = ['trip_count', 'month', 'year', 'day_of_month', 'PULocationID']))
  newReport = pd.DataFrame([{
    "BaseMonths": current_train_start.strftime("%B") + "-" + (current_train_end - relativedelta(months=1)).strftime("%B"),
    "TestMonths": current_train_end.strftime("%B") ,
    "Report": eval
}])

  reports = pd.concat([reports, newReport], ignore_index = True)


  # Step 2: recombine to one-hot encode consistently, then re-split

  combined = pl.concat([train_df.with_columns(pl.lit('train').alias('_split')),
                       test_df.with_columns(pl.lit('test').alias('_split'))])


  combined = combined.to_dummies(Catcolumns, drop_first=True)

  combined = combined.with_columns([
    (2 * np.pi * pl.col('hour') / 24).sin().alias('hour_sin'),
    (2 * np.pi * pl.col('hour') / 24).cos().alias('hour_cos')
  ])

  test_timestamps = test_df['tpep_pickup_datetime'].to_numpy()


  combined = combined.drop('tpep_pickup_datetime')
  train_df = combined.filter(pl.col('_split') == 'train').drop('_split')
  test_df = combined.filter(pl.col('_split') == 'test').drop('_split')


  train_df = train_df.to_pandas()
  test_df = test_df.to_pandas()

  X_train, y_train = train_df.drop(columns = 'trip_count'), train_df['trip_count']
  X_test, y_test = test_df.drop(columns = 'trip_count'), test_df['trip_count']


  X_train = X_train.drop(columns=['hour'])
  X_test = X_test.drop(columns=['hour'])





  #Using and Testing on Chosen Model of XGBoost


  xgModel = xgb.XGBRegressor(
    tree_method = 'hist',
    enable_categorical = True,
    n_estimators=200,     # Maximum number of sequential trees
    max_depth=6,          # Depth of each tree
    learning_rate=0.02,    # Step size shrinkage (eta)

  )

  xgModel.fit(X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100)

  y_predX = xgModel.predict(X_test)

  train_pred = xgModel.predict(X_train)
  train_resids = y_train - train_pred
  train_mae = mean_absolute_error(y_train, train_pred)

  dynamic_delta = train_mae * 0.2
  dynamic_lambda = train_mae * 500

  res = np.abs(y_predX - y_test)

  #Concept Drift using Page Hinkley on Residuals
  sort_idx = np.argsort(test_timestamps)
  sorted_resids = res.to_numpy()[sort_idx]

  concept_drifted, drifted_at_row = concept_drift_with_warmup(
    train_res=train_resids,
    test_res=sorted_resids,
    delta=dynamic_delta,
    lambda_thresh=dynamic_lambda
)

  if concept_drifted:
    drift_time = test_timestamps[sort_idx][drifted_at_row]
    print(f"⚠️ CONCEPT DRIFT DETECTED in month {current_train_end.strftime('%B')}!")
    print(f"Model relationship failed around timestamp: {drift_time}")
    print(f"Prior to drift, mean error was stable, but cumulative loss exceeded threshold.")
  else:
      print(f"✅ Concept stable for {current_train_end.strftime('%B')}. Model patterns hold.")


  mse = mean_squared_error(y_true = y_test, y_pred = y_predX)
  rmse = np.sqrt(mse)
  mae = mean_absolute_error(y_test, y_predX)
  r2 = r2_score(y_test, y_predX)
  print(f"RMSE: {rmse:.4f}")
  print(f"R² Score: {r2:.4f}")
  print(f"Mean Absolute Residual (Matches MAE): {res.mean():.4f}")
  print(f"Max Absolute Residual Error: {res.max():.4f}")


  window_predictions_df = pl.DataFrame({
      "timestamp": test_timestamps,
      "test_month": current_train_end.strftime("%Y-%m"),
      "actuals": y_test.to_numpy(),
      "predictions": y_predX.astype(np.float64)
  })

  results.append(window_predictions_df)

  metrics_summary.append({
      "test_month": current_train_end.strftime("%Y-%m"),
      "rmse": rmse,
      "mae": mae,
      "r2": r2,
      "max_residual": res.max(),
      "mean_residual": res.mean()
  })

  current_train_start = current_train_start + relativedelta(months=1)


forecast_df = pl.concat(results)
forecast_df = forecast_df.with_columns(
    (pl.col("actuals") - pl.col("predictions")).alias("residual")
)

summary_df = pl.DataFrame(metrics_summary)
print("--- RAW PREDICTIONS FRAME (thousands of rows) ---")
print(forecast_df.head())

print(summary_df)




There has been a data drift with 25.0% of the features thus a covariate shift
Occured when comparing base months of January-April to test month of May
             Feature  K-S Distance (D)    Raw p-value  Feature Drifted
0               hour          0.011480   7.027617e-12            False
1  trip_count_lag_1h          0.041985  1.202003e-153            False
2  trip_count_lag_2h          0.041842  1.330202e-152            False
3       loc_hour_avg          0.057061  1.586080e-283             True
[0]	validation_0-rmse:75.46053
[100]	validation_0-rmse:20.41724
[199]	validation_0-rmse:15.87421
⚠️ CONCEPT DRIFT DETECTED in month May!
Model relationship failed around timestamp: 2025-05-01T21:00:00.000000
Prior to drift, mean error was stable, but cumulative loss exceeded threshold.
RMSE: 15.8742
R² Score: 0.9573
Mean Absolute Residual (Matches MAE): 6.5914
Max Absolute Residual Error: 403.2253
[0]	validation_0-rmse:72.09410
[100]	validation_0-rmse:17.87175
[199]	validation_0-rmse:14.08

In [45]:
reports.iloc[4]

,4
BaseMonths,May-August
TestMonths,September
Report,<evidently.core.report.Snapshot object at 0x7a...


In [46]:
reports.iloc[4]["Report"]

Output hidden; open in https://colab.research.google.com to view.